In [16]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

print("1. Đang tải và GIẢI NÉN dữ liệu...")
df_raw = pd.read_csv('../data/raw/train.csv').sample(3000, random_state=42)
df_raw = df_raw.dropna(subset=['comment', 'label'])

parsed_data = []

# Vòng lặp giải nén: Tách '{ASPECT#SENTIMENT}' thành từng dòng riêng
for index, row in df_raw.iterrows():
    comment = str(row['comment']).strip()
    label_str = str(row['label'])
    
    # Dùng Regex để tìm tất cả các cặp {Khía_cạnh#Cảm_xúc}
    matches = re.findall(r'\{([^#\}]+)#([^#\}]+)\}', label_str)
    
    for aspect, sentiment in matches:
        parsed_data.append({
            'comment': comment,
            'aspect': aspect,
            'sentiment': sentiment
        })

# Tạo DataFrame mới từ dữ liệu đã giải nén
df = pd.DataFrame(parsed_data)

print(f"=> Đã giải nén thành công {len(df)} cặp (Bình luận - Khía cạnh)!")

# Chuyển đổi nhãn tiếng Anh sang số
label_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
df['label'] = df['sentiment'].map(label_map)

# Dọn dẹp lỗi nếu có và ép kiểu số nguyên
df = df.dropna(subset=['comment', 'aspect', 'label'])
df['label'] = df['label'].astype(int)

# Ghép câu bình luận và khía cạnh lại với nhau chuẩn PhoBERT (Dùng </s></s>)
texts = (df['comment'] + " </s></s> " + df['aspect']).tolist()
labels = df['label'].tolist()

# Chia tập train/val để đánh giá
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.1, random_state=42)

print(f"Tổng số câu hợp lệ đưa vào Train: {len(texts)}")
print("Mẫu dữ liệu đầu tiên trông như thế này:", texts[0], "--> Nhãn:", labels[0])

1. Đang tải và GIẢI NÉN dữ liệu...
=> Đã giải nén thành công 9335 cặp (Bình luận - Khía cạnh)!
Tổng số câu hợp lệ đưa vào Train: 9335
Mẫu dữ liệu đầu tiên trông như thế này: Lấy máy về 2 ngày bị lỗi vân tay. Nhận diện khuôn mặt cũng chậm. Không đáng giá tiền tí nào. Chán😟 </s></s> FEATURES --> Nhãn: 0


In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset
import os

print("2. Đang tải Tokenizer và 'Não bộ' PhoBERT gốc...")
MODEL_NAME = "vinai/phobert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
# 3 Nhãn: 0 (Tiêu cực), 1 (Trung tính), 2 (Tích cực)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# --- TẠO LỚP DỮ LIỆU CHUẨN HÓA CHO AI ---
class ABSADataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64): 
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        # ĐÃ SỬA LỖI Ở ĐÂY: Dùng trực tiếp tokenizer() chuẩn mới nhất thay vì encode_plus
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

print("3. Đang đóng gói dữ liệu mớm cho mô hình...")
train_dataset = ABSADataset(train_texts, train_labels, tokenizer)
val_dataset = ABSADataset(val_texts, val_labels, tokenizer)

# --- CẤU HÌNH HUẤN LUYỆN ---
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("4. BẮT ĐẦU HUẤN LUYỆN (Hãy kiên nhẫn, thanh tiến trình sắp hiện ra)...")
trainer.train()

print("5. LƯU MÔ HÌNH THÀNH PHẨM...")
os.makedirs('../models/phobert_absa', exist_ok=True)

trainer.save_model('../models/phobert_absa')
tokenizer.save_pretrained('../models/phobert_absa')

print("Mô hình đã được lưu thành công tại: models/phobert_absa")

2. Đang tải Tokenizer và 'Não bộ' PhoBERT gốc...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 35211.71it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing f

3. Đang đóng gói dữ liệu mớm cho mô hình...
4. BẮT ĐẦU HUẤN LUYỆN (Hãy kiên nhẫn, thanh tiến trình sắp hiện ra)...


Epoch,Training Loss,Validation Loss
1,0.562298,0.564782


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.00it/s]


5. LƯU MÔ HÌNH THÀNH PHẨM...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.58it/s]

Mô hình đã được lưu thành công tại: models/phobert_absa
